In [7]:
!pip install -q \
    ollama \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-ollama \
    chromadb \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/7

In [8]:
import ollama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_1222/4180757495.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
models_dict = ollama.list()

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [5]:
model_name = models_dict.models[11].model #gemma3:4b
print(model_name)

NameError: name 'models_dict' is not defined

In [4]:
# Célula 2: Importação das bibliotecas e configuração do pipeline


ModuleNotFoundError: No module named 'langchain_community'

In [ ]:

# ==========================================
# 2. EMBEDDINGS E BANCO DE VETORES
# ==========================================
diretorio_db = "./data_documents"
# Criando IDs únicos baseados no nome do arquivo e número do pedaço


embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 3. Carregue o banco existente
vectorstore = Chroma(
    persist_directory=diretorio_db,
    embedding_function=embeddings
)



/tmp/ipykernel_211632/801421038.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [ ]:

#preparação para a pergunta com similaridade de 3 trechos mais relevantes parecidos
# Configura o buscador para trazer os 3 blocos mais relevantes para a pergunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:

# ==========================================
# 3. CONFIGURAÇÃO DO LLM E PROMPT RAG
# ==========================================

# temperature indica o nível de criatividade da LLM, ou grau de liberdade
llm = ChatOllama(model=model_name, temperature=0)

template_rag = """
[CONTEXTO]
Utilize estritamente as informações extraídas do regulamento de estágio da UFRB em PDF fornecidas abaixo para responder à pergunta do usuário de forma clara e direta.
Se a resposta não puder ser encontrada nos fragmentos, diga apenas que a informação não consta no documento.

FRAGMENTOS DO PDF:
{context}

[PERGUNTA]
{question}

[RESPOSTA]
"""
prompt = ChatPromptTemplate.from_template(template_rag)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construção da cadeia RAG
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Sistema de RAG com suporte a PDF pronto!")

Sistema de RAG com suporte a PDF pronto!


In [ ]:
# Célula 3: Consulta ao documento PDF
pergunta = "Quais documentos são necessários para redução de carga horária?"


resposta = """I-Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);
II - Três últimos contracheques (apenas a parte que indica nome, matrícula e mês do pagamento);
III - Atestado de frequência da escola, discriminando nível de ensino, ano, disciplina, turno e carga horária;
IV - Relatório da Coordenação de Área, ou Coordenação Pedagógica ou da Direção, avaliando o perfil profissional do professor em formação."""



In [ ]:
import time
import metrics

/home/workstationvialab/miniconda3/envs/llms_ollama/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


inicio = time.time()
resposta_final = rag_chain.invoke(pergunta)
fim = time.time()

print(resposta_final)


metricas = metrics.calcular_metricas_operacionais(
    prompt=pergunta,
    resposta=resposta_final,
    tempo_inicio=inicio,
    tempo_fim=fim
)

metricas

Para efeito de redução de carga horária de Estágio Curricular Supervisionado, o discente deverá apresentar:

I - Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);
II - Três últimos contracheques (apenas a parte que indica nome, matrícula e mês do pagamento);
III - Atestado de frequência da escola, discriminando nível de ensino, ano, disciplina, turno e carga horária;
IV - Relatório da Coordenação de Área, ou Coordenação Pedagógica ou da Direção, avaliando o perfil profissional do professor em formação.


{'tokens_totais': 177,
 'latencia_segundos': 11.54,
 'tokens_por_segundo': 14.12,
 'custo_estimado_usd': 0.000266}

In [ ]:
rouge = metrics.calcular_rouge(resposta_final, resposta)
rouge

{'rouge1_f1': 0.9053, 'rouge2_f1': 0.9043, 'rougeL_f1': 0.9053}

In [ ]:
eval_judge = metrics.llmAsJudge(pergunta, resposta_final, resposta, models_dict.models[2].model)

Evaluating: 100%|██████████| 1/1 [01:24<00:00, 84.03s/it]


In [ ]:
eval_judge

{'faithfulness': 1.0000}

In [ ]:
# Célula 3: Consulta ao documento PDF
pergunta1 = "Pode ser solicitado dispensa do estágio supervisionado obrigatório?"


resposta1 = """Sob nenhuma hipótese o estudante será dispensado do Estágio Supervisionado
Obrigatório, nem mesmo será permitida a realização de atividades domiciliares por
motivo de doença ou licença maternidade. Nestes casos, o estudante não poderá se
matricular. Entretanto, caso ele esteja cursando o componente curricular Estágio
Supervisionado deverá solicitar o trancamento do mesmo e se matricular em outro
semestre, no prazo estipulado pela Universidade."""


inicio = time.time()
resposta_final1 = rag_chain.invoke(pergunta1)
fim = time.time()

print(resposta_final1)


metricas1 = metrics.calcular_metricas_operacionais(
    prompt=pergunta1,
    resposta=resposta_final1,
    tempo_inicio=inicio,
    tempo_fim=fim
)

print(metricas1)

rouge1 = metrics.calcular_rouge(resposta_final1, resposta1)
print(rouge1)

eval_judge1 = metrics.llmAsJudge(pergunta1, resposta_final1, resposta1, models_dict.models[2].model)
print(eval_judge1)

A informação não consta no documento.
{'tokens_totais': 24, 'latencia_segundos': 2.08, 'tokens_por_segundo': 4.32, 'custo_estimado_usd': 3.6e-05}
{'rouge1_f1': 0.1282, 'rouge2_f1': 0.0263, 'rougeL_f1': 0.1282}


Evaluating: 100%|██████████| 1/1 [00:36<00:00, 36.17s/it]


{'faithfulness': 0.0000}
